# LeetCode 2275 — Largest Combination With Bitwise AND Greater Than Zero bằng Python

Link đề: https://leetcode.com/problems/largest-combination-with-bitwise-and-greater-than-zero/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Bit Manipulation, Counting

Skeleton LeetCode:

```python
class Solution:
    def largestCombination(self, candidates: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Hiểu phép **AND bit** trên nhiều số: kết quả có bit `b` bằng 1 ⇔ **mọi** số đều có bit `b`.
2. Chuyển bài "tập con lớn nhất có AND > 0" thành "**bit nào xuất hiện ở nhiều số nhất**" — đếm theo từng cột bit.
3. Thành thạo thao tác `x >> b & 1`, `bit_length()`, `bin(x)` và cách đếm bit theo cột hiệu quả trong Python.
4. Chứng minh: tập có AND > 0 ⇔ tồn tại bit chung; tập lớn nhất = tập mọi số chứa một bit cố định.
5. So sánh vét cạn tập con (2ⁿ) với đếm bit O(n·B), và các biến thể (AND ≥ k, OR, AND bằng đúng một giá trị).

## 2. Đề bài & ví dụ

AND bit của một mảng là AND của mọi phần tử. Cho mảng số nguyên dương `candidates`. Xét AND của **mọi tổ hợp** (tập con khác rỗng) các phần tử.

Trả về **kích thước lớn nhất** của một tổ hợp có AND **> 0**.

**Ví dụ 1:** `candidates = [16,17,71,62,12,24,14]` → `4`: `[16,17,62,24]` có AND = 16 > 0; không tổ hợp 5 phần tử nào có AND > 0.

**Ví dụ 2:** `candidates = [8,8]` → `2`.

**Ràng buộc:** `1 ≤ len(candidates) ≤ 10⁵`, `1 ≤ candidates[i] ≤ 10⁷` (< 2²⁴).

In [1]:
from typing import List
from functools import reduce
from itertools import combinations
import operator, random, time

examples = [([16, 17, 71, 62, 12, 24, 14], 4), ([8, 8], 2)]
for c, exp in examples:
    print(c, "-> kỳ vọng", exp)
    for x in c: print(f"   {x:>3} = {x:08b}")

[16, 17, 71, 62, 12, 24, 14] -> kỳ vọng 4
    16 = 00010000
    17 = 00010001
    71 = 01000111
    62 = 00111110
    12 = 00001100
    24 = 00011000
    14 = 00001110
[8, 8] -> kỳ vọng 2
     8 = 00001000
     8 = 00001000


## 3. Kiến thức nền

### 3.1 AND bit trên nhiều số

In [2]:
print(1 & 5 & 3, reduce(operator.and_, [16, 17, 62, 24]), reduce(operator.and_, [16, 17, 71, 62, 24]))
for x in [16, 17, 62, 24]: print(f"{x:08b}")
print(f"{16 & 17 & 62 & 24:08b} <- chỉ bit 4 (giá trị 16) chung cho cả bốn số")

1 16 0
00010000
00010001
00111110
00011000
00010000 <- chỉ bit 4 (giá trị 16) chung cho cả bốn số


### 3.2 Lấy bit thứ `b`: `x >> b & 1`

`>>` dịch phải, `& 1` lấy bit thấp nhất. Độ ưu tiên: `>>` mạnh hơn `&` nên `x >> b & 1 = (x >> b) & 1`.

In [3]:
x = 62
print([x >> b & 1 for b in range(8)], "| bin:", bin(x), "| bit_length:", x.bit_length())

[0, 1, 1, 1, 1, 1, 0, 0] | bin: 0b111110 | bit_length: 6


### 3.3 Đếm theo cột bit

Với mỗi vị trí bit `b`, đếm số phần tử có bit đó: `cnt[b] = Σ (x >> b & 1)`.

In [4]:
c = [16, 17, 71, 62, 12, 24, 14]
cnt = [sum(x >> b & 1 for x in c) for b in range(8)]
print("cnt theo bit 0..7:", cnt)

cnt theo bit 0..7: [2, 3, 4, 4, 4, 1, 1, 0]


### 3.4 Số bit cần xét

`10⁷ < 2²⁴` ⇒ chỉ cần bit `0..23`. Tổng quát: `max(candidates).bit_length()` bit.

In [5]:
print((10**7).bit_length(), 2**23 <= 10**7 < 2**24)

24 True


### 3.5 Nhìn mảng như một **ma trận bit**

Xếp mỗi số thành một hàng nhị phân; cột `b` là bit `b` của mọi số. Bài toán trở thành: **cột nào có nhiều số 1 nhất?** Đây là cách nhìn "chuyển vị" (transpose) giúp tách bài toán thành 24 bài con độc lập.

In [6]:
c = [16, 17, 71, 62, 12, 24, 14]
W = max(c).bit_length()
rows = [[x >> b & 1 for b in range(W - 1, -1, -1)] for x in c]
print("     bit:", " ".join(str(b) for b in range(W - 1, -1, -1)))
for x, r in zip(c, rows): print(f"{x:>8}:", " ".join(map(str, r)))
cols = list(zip(*rows))
print("   tổng:", " ".join(str(sum(col)) for col in cols))

     bit: 6 5 4 3 2 1 0
      16: 0 0 1 0 0 0 0
      17: 0 0 1 0 0 0 1
      71: 1 0 0 0 1 1 1
      62: 0 1 1 1 1 1 0
      12: 0 0 0 1 1 0 0
      24: 0 0 1 1 0 0 0
      14: 0 0 0 1 1 1 0
   tổng: 1 1 4 4 4 3 2


## 4. Ý tưởng nền tảng / trực giác

AND của một tổ hợp > 0 ⇔ kết quả có **ít nhất một** bit 1 ⇔ có một vị trí bit `b` mà **mọi** số trong tổ hợp đều có bit `b`.

Vậy: chọn một bit `b`, lấy **tất cả** các số có bit `b` ⇒ AND của chúng chắc chắn có bit `b` ⇒ > 0. Kích thước = `cnt[b]`. Đáp án = `max_b cnt[b]`.

Tính tay ví dụ 1 (bảng bit, cột là bit 0..6):

| số | b6 | b5 | b4 | b3 | b2 | b1 | b0 |
|---|---|---|---|---|---|---|---|
| 16 | 0 | 0 | 1 | 0 | 0 | 0 | 0 |
| 17 | 0 | 0 | 1 | 0 | 0 | 0 | 1 |
| 71 | 1 | 0 | 0 | 0 | 1 | 1 | 1 |
| 62 | 0 | 1 | 1 | 1 | 1 | 1 | 0 |
| 12 | 0 | 0 | 0 | 1 | 1 | 0 | 0 |
| 24 | 0 | 0 | 1 | 1 | 0 | 0 | 0 |
| 14 | 0 | 0 | 0 | 1 | 1 | 1 | 0 |
| **cnt** | 1 | 1 | **4** | **4** | **4** | 3 | 2 |

Max = 4 (bit 4: `{16, 17, 62, 24}`; bit 3: `{62, 12, 24, 14}`; bit 2: `{71, 62, 12, 14}`).

In [7]:
c = examples[0][0]
for b in range(7):
    group = [x for x in c if x >> b & 1]
    print(f"bit {b}: {group} -> AND = {reduce(operator.and_, group) if group else '-'}")

bit 0: [17, 71] -> AND = 1
bit 1: [71, 62, 14] -> AND = 6
bit 2: [71, 62, 12, 14] -> AND = 4
bit 3: [62, 12, 24, 14] -> AND = 8
bit 4: [16, 17, 62, 24] -> AND = 16
bit 5: [62] -> AND = 62
bit 6: [71] -> AND = 71


### 4.1 AND chỉ "mất" bit, không bao giờ "được" bit

Thêm một số vào tập: `AND_mới = AND_cũ & x` — mỗi bit của kết quả chỉ có thể từ 1 xuống 0. Vì vậy càng thêm nhiều số, tập bit "sống sót" càng ít; tập lớn nhất có AND > 0 phải giữ lại **ít nhất một** bit sống sót tới cùng — và để lớn nhất, ta chọn bit đó trước rồi lấy mọi số có nó.

In [8]:
seq = [62, 24, 16, 17, 71]
a = -1
for x in seq:
    a &= x
    print(f"thêm {x:>3} ({x:08b}) -> AND = {a:>3} ({a & 0xFF:08b})")

thêm  62 (00111110) -> AND =  62 (00111110)
thêm  24 (00011000) -> AND =  24 (00011000)
thêm  16 (00010000) -> AND =  16 (00010000)
thêm  17 (00010001) -> AND =  16 (00010000)
thêm  71 (01000111) -> AND =   0 (00000000)


Thêm `71` (không có bit 4) làm bit cuối cùng còn sống (bit 4) chết ⇒ AND = 0.

## 5. Cách 1 — Brute Force / baseline: thử mọi tập con

In [9]:
def brute(candidates):
    n = len(candidates); best = 0
    for mask in range(1, 1 << n):
        a = -1; size = 0
        for i in range(n):
            if mask >> i & 1:
                a &= candidates[i]; size += 1
        if a > 0 and size > best: best = size
    return best

for c, exp in examples:
    got = brute(c)
    print(got, "OK" if got == exp else "SAI")

4 OK
2 OK


`a = −1` (mọi bit 1 trong biểu diễn bù hai của Python) là phần tử trung hòa của AND.

**Độ phức tạp:** O(2ⁿ · n) — n = 20 đã ~2·10⁷ phép; n = 10⁵ là không tưởng (2^100000 tập con).

**Nút thắt:** ta thử mọi tập, trong khi điều kiện AND > 0 chỉ phụ thuộc vào việc "có bit nào chung". Đếm theo bit loại bỏ hoàn toàn việc liệt kê tập.

### 5.1 Baseline tham lam (sai): thêm dần số "giữ được AND > 0"

Duyệt các số, giữ AND hiện tại; nếu thêm số mới vẫn > 0 thì thêm. Có vẻ hợp lý nhưng phụ thuộc thứ tự — xem Lỗi 2 ở mục 13.

## 6. Ý tưởng thuật toán / bước đột phá

**Định lý:** `max{|S| : AND(S) > 0} = max_b cnt[b]`.

- (≥) Với bit `b`, tập `S_b = {x : x có bit b}` có AND chứa bit `b` ⇒ > 0, và `|S_b| = cnt[b]`.
- (≤) Nếu `AND(S) > 0` thì có bit `b` thuộc `AND(S)` ⇒ mọi `x ∈ S` có bit `b` ⇒ `S ⊆ S_b` ⇒ `|S| ≤ cnt[b]`.

**Quy tắc:** với mỗi bit `b ∈ [0, 24)`: đếm `cnt[b]`; đáp án = max.

**Hai cách tổ chức vòng lặp:**
1. **Theo bit ngoài, theo số trong:** `for b: sum(x >> b & 1 for x in candidates)` — 24 lượt qua mảng.
2. **Theo số ngoài, theo bit trong:** với mỗi `x`, chỉ duyệt các bit **bật** của `x` (`while x: low = x & −x …`) — tổng số bước = tổng số bit 1.

Cả hai O(n · B) với `B = 24`.

### 6.1 Vì sao không cần xét **tổ hợp các bit**?

Có thể nghĩ: "tập tốt nhất có thể dùng bit 3 **và** bit 4 cùng lúc thì sao?" — Nếu mọi phần tử của tập có cả bit 3 và bit 4 thì tập đó ⊆ `S_3` và ⊆ `S_4` ⇒ kích thước ≤ `min(cnt[3], cnt[4])` ≤ `max_b cnt[b]`. Yêu cầu **nhiều** bit chung chỉ làm tập **nhỏ đi**. Vậy một bit là đủ — đây là lý do bài "trông như" bài tập con mũ lại giải được bằng 24 phép đếm.

**So sánh khối lượng công việc:** vét cạn xét `2ⁿ − 1` tập; đếm bit xét `24·n` cặp (số, bit). Với `n = 10⁵`: `2^100000` (không tưởng) so với `2,4·10⁶`.

In [10]:
c = [16, 17, 71, 62, 12, 24, 14]
cnt = {b: sum(x >> b & 1 for x in c) for b in range(7)}
for b1 in range(7):
    for b2 in range(b1 + 1, 7):
        both = sum(1 for x in c if x >> b1 & 1 and x >> b2 & 1)
        assert both <= min(cnt[b1], cnt[b2])
print("mọi cặp bit: số phần tử có cả hai bit <= min(cnt) — yêu cầu thêm bit chỉ làm tập nhỏ đi")

mọi cặp bit: số phần tử có cả hai bit <= min(cnt) — yêu cầu thêm bit chỉ làm tập nhỏ đi


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Theo từng bit (cách doocs)

In [11]:
def per_bit(candidates):
    ans = 0
    for b in range(max(candidates).bit_length()):
        ans = max(ans, sum(x >> b & 1 for x in candidates))
    return ans

for c, exp in examples:
    got = per_bit(c)
    print(got, "OK" if got == exp else "SAI")

4 OK
2 OK


**Độ phức tạp cách 2:** O(n · B), B ≤ 24 ⇒ ~2,4·10⁶ phép; bộ nhớ O(1). Dùng `max(candidates).bit_length()` thay vì cố định 24 giúp bớt vòng khi các số nhỏ.

### 7.2 Cách 3 — Theo từng số, chỉ duyệt bit bật

In [12]:
def per_number(candidates):
    cnt = [0] * 24
    for x in candidates:
        while x:
            low = x & -x                     # bit thấp nhất đang bật
            cnt[low.bit_length() - 1] += 1
            x ^= low                         # tắt bit đó
    return max(cnt)

for c, exp in examples:
    got = per_number(c)
    print(got, "OK" if got == exp else "SAI")

4 OK
2 OK


`x & −x` cô lập bit thấp nhất đang bật (mẹo bù hai: `−x = ~x + 1`, nên mọi bit trên bit thấp nhất bị đảo, bit thấp nhất giữ nguyên). Số vòng `while` = số bit 1 của `x` (≤ 24, trung bình ~12 với số ngẫu nhiên).

### 7.3 Cách 4 — Đếm qua chuỗi nhị phân `bin(x)`

In [13]:
def via_bin_strings(candidates):
    cnt = [0] * 24
    for x in candidates:
        s = bin(x)[2:][::-1]                 # bit 0 ở đầu chuỗi
        for b, ch in enumerate(s):
            if ch == "1": cnt[b] += 1
    return max(cnt)

for c, exp in examples:
    print(via_bin_strings(c), "OK" if via_bin_strings(c) == exp else "SAI")

4 OK
2 OK


### 7.4 Cách 5 — Lọc dần theo bit với `sum(map(...))` (tối ưu hằng số Python)

`sum(1 for x in c if x & m)` với mặt nạ `m = 1 << b`; viết bằng `map` + hàm tích hợp để vòng lặp chạy nhanh hơn.

In [14]:
def masked_counts(candidates):
    return max(sum(1 for x in candidates if x & (1 << b)) for b in range(24))

for c, exp in examples:
    print(masked_counts(c), "OK" if masked_counts(c) == exp else "SAI")

4 OK
2 OK


### 7.5 Cách 6 — Gom số trùng bằng `Counter` trước (khi nhiều giá trị lặp)

Nếu mảng có nhiều số giống nhau, đếm tần suất trước rồi đếm bit theo **giá trị phân biệt** có trọng số (mỗi giá trị đóng góp đúng tần suất của nó vào các cột bit mà nó có).

In [15]:
from collections import Counter
def with_counter(candidates):
    freq = Counter(candidates); cnt = [0] * 24
    for v, f in freq.items():
        for b in range(v.bit_length()):
            if v >> b & 1: cnt[b] += f
    return max(cnt)

for c, exp in examples:
    print(with_counter(c), "OK" if with_counter(c) == exp else "SAI")

4 OK
2 OK


## 8. TRACE từng bước

Trace cách 2 trên ví dụ 1 — in nhóm số và số đếm mỗi bit.

In [16]:
def trace(candidates):
    ans = 0
    for b in range(max(candidates).bit_length()):
        grp = [x for x in candidates if x >> b & 1]; ans = max(ans, len(grp))
        print(f"bit {b} (giá trị {1 << b:>3}): {len(grp)} số {grp} -> ans = {ans}")
    return ans

print("Kết quả:", trace([16, 17, 71, 62, 12, 24, 14]))

bit 0 (giá trị   1): 2 số [17, 71] -> ans = 2
bit 1 (giá trị   2): 3 số [71, 62, 14] -> ans = 3
bit 2 (giá trị   4): 4 số [71, 62, 12, 14] -> ans = 4
bit 3 (giá trị   8): 4 số [62, 12, 24, 14] -> ans = 4
bit 4 (giá trị  16): 4 số [16, 17, 62, 24] -> ans = 4
bit 5 (giá trị  32): 1 số [62] -> ans = 4
bit 6 (giá trị  64): 1 số [71] -> ans = 4
Kết quả: 4


Trace cách 3 cho một số, thấy từng bit bật được tách ra:

In [17]:
x = 62
while x:
    low = x & -x
    print(f"x = {x:08b}, x & -x = {low:08b} (bit {low.bit_length() - 1}) -> tắt -> {x ^ low:08b}")
    x ^= low

x = 00111110, x & -x = 00000010 (bit 1) -> tắt -> 00111100
x = 00111100, x & -x = 00000100 (bit 2) -> tắt -> 00111000
x = 00111000, x & -x = 00001000 (bit 3) -> tắt -> 00110000
x = 00110000, x & -x = 00010000 (bit 4) -> tắt -> 00100000
x = 00100000, x & -x = 00100000 (bit 5) -> tắt -> 00000000


## 9. Code nộp LeetCode

In [18]:
from typing import List

class Solution:
    def largestCombination(self, candidates: List[int]) -> int:
        ans = 0
        for b in range(24):                       # candidates[i] <= 10^7 < 2^24
            mask = 1 << b
            cnt = 0
            for x in candidates:
                if x & mask:
                    cnt += 1
            if cnt > ans:
                ans = cnt
        return ans

In [19]:
sol = Solution()
for c, exp in examples:
    got = sol.largestCombination(c)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

4 | kỳ vọng 4 | OK
2 | kỳ vọng 2 | OK


## 10. Vì sao thuật toán đúng

Đã chứng minh ở mục 6 (hai chiều ≥ và ≤). Diễn giải thêm:
- **AND không tăng** khi thêm phần tử: `AND(S ∪ {x}) = AND(S) & x ≤ AND(S)` (theo nghĩa tập bit). Vì vậy "tập lớn nhất" bị giới hạn bởi các bit còn sống sót.
- Một tập có AND > 0 có thể có **nhiều** bit chung; ta chỉ cần **một** bit để chứng tỏ AND > 0, và với bit đó, tập `S_b` là **tập lớn nhất** chứa bit đó (thêm số nào không có bit `b` thì bit `b` chết).
- Có thể xảy ra `S_b` có AND gồm nhiều bit hơn `b` — không sao, AND vẫn > 0.

Mọi `candidates[i] ≥ 1` ⇒ mỗi số có ít nhất một bit ⇒ đáp án ≥ 1. ∎

**Ví dụ tập có nhiều bit chung:** `[62, 14, 12]` = `111110, 001110, 001100` có AND = `001100` = 12 (hai bit chung 2 và 3). Nó nằm trong `S_2` (4 phần tử) và `S_3` (4 phần tử) của ví dụ 1 — cả hai đều lớn hơn, đúng như lập luận ở mục 6.1.

**Ghi nhớ quan trọng:** cấu trúc "tối ưu trên mọi tập con" sụp đổ thành "tối ưu trên 24 ứng viên" nhờ hai tính chất của AND: (1) **đơn điệu** (thêm phần tử không thể bật lại bit đã tắt) và (2) **tách theo bit** (điều kiện AND > 0 là "tồn tại bit", tức hợp của các điều kiện từng bit).

## 11. Phân tích Time / Space Complexity

`n ≤ 10⁵`, `B = 24`.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi tập con | O(2ⁿ · n) | O(1) |
| 2. Theo bit | O(n · B) | O(1) |
| 3. Theo số, bit bật | O(tổng số bit 1) ≤ O(n · B) | O(B) |
| 4. Chuỗi nhị phân | O(n · B) | O(B) |
| 5. Mặt nạ + generator | O(n · B) | O(1) |
| 6. Counter trước | O(n + D · B) | O(D) |

In [20]:
rng = random.Random(2275)
big = [rng.randint(1, 10**7) for _ in range(10**5)]
dup = [rng.choice([3, 5, 6, 7, 1 << 20]) for _ in range(10**5)]
for data, nm in [(big, "ngẫu nhiên"), (dup, "nhiều trùng")]:
    res = []
    for name, f in [("per_bit", per_bit), ("per_number", per_number), ("via_bin_strings", via_bin_strings),
                    ("masked_counts", masked_counts), ("with_counter", with_counter), ("Solution", Solution().largestCombination)]:
        t0 = time.perf_counter(); res.append(f(data)); dt = time.perf_counter() - t0
        print(f"{nm:<11} {name:<16} -> {res[-1]}  {dt*1000:6.0f} ms")
    assert len(set(res)) == 1

ngẫu nhiên  per_bit          -> 50205      89 ms


ngẫu nhiên  per_number       -> 50205     117 ms


ngẫu nhiên  via_bin_strings  -> 50205     153 ms


ngẫu nhiên  masked_counts    -> 50205     122 ms


ngẫu nhiên  with_counter     -> 50205     110 ms
ngẫu nhiên  Solution         -> 50205      79 ms


nhiều trùng per_bit          -> 60242      70 ms
nhiều trùng per_number       -> 60242      15 ms
nhiều trùng via_bin_strings  -> 60242      38 ms


nhiều trùng masked_counts    -> 60242      75 ms
nhiều trùng with_counter     -> 60242       3 ms
nhiều trùng Solution         -> 60242      43 ms


**Nhận xét về hằng số trong Python:** cả sáu cách đều O(n·B), nhưng tốc độ thực tế khác nhau vài lần: vòng lặp Python thuần (`for x in ...: if x & mask`) tốn ~50 ns mỗi bước; `sum(generator)` nhanh hơn chút; `Counter` thắng lớn khi có nhiều giá trị trùng (chỉ đếm bit trên giá trị phân biệt). Khi n = 10⁵ thì mọi cách đều dưới 1 giây — chọn cách **dễ đọc và khó sai** nhất (cách 2 / bản nộp).

**Bộ nhớ:** chỉ cần một mảng 24 bộ đếm (hoặc một biến) — không phụ thuộc n.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [21]:
ALL = {"brute": brute, "per_bit": per_bit, "per_number": per_number, "via_bin_strings": via_bin_strings,
       "masked_counts": masked_counts, "with_counter": with_counter, "Solution": Solution().largestCombination}
edge = examples + [
    ([1], 1),
    ([1, 2, 4, 8], 1),                  # không có bit chung giữa hai số bất kỳ
    ([3, 3, 3], 3),
    ([1, 3, 5, 7], 4),                  # bit 0 chung
    ([10**7], 1),
    ([2**23, 2**23 + 1, 1], 2),
    ([6, 5, 3], 2),                     # mỗi cặp có một bit chung, cả ba thì không
]
for c, exp in edge:
    for name, f in ALL.items():
        assert f(list(c)) == exp, (name, c, f(c), exp)
print("Edge case OK:", len(edge))

Edge case OK: 9


Case `[6, 5, 3]` = `110, 101, 011`: mọi **cặp** có một bit chung (6&5 = 4, 6&3 = 2, 5&3 = 1) nhưng `6&5&3 = 0` ⇒ đáp án 2. Đây là ví dụ cho thấy "có bit chung từng cặp" **không** kéo theo "có bit chung cả nhóm".

### 12.2 Đối chiếu ngẫu nhiên với vét cạn tập con (n ≤ 11)

In [22]:
rng = random.Random(42)
for _ in range(300):
    n = rng.randint(1, 11); hi = rng.choice([7, 31, 255, 10**7])
    c = [rng.randint(1, hi) for _ in range(n)]
    ref = brute(c)
    for name, f in ALL.items():
        assert f(list(c)) == ref, (name, c)
print("300 case ngẫu nhiên OK")

300 case ngẫu nhiên OK

### 12.3 Tính chất: tập tạo ra từ bit tối ưu thực sự có AND > 0; thêm bất kỳ số ngoài tập làm AND = 0 hoặc không tăng kích thước hợp lệ

In [23]:
rng = random.Random(7)
for _ in range(300):
    c = [rng.randint(1, 255) for _ in range(rng.randint(1, 40))]
    best_b = max(range(8), key=lambda b: sum(x >> b & 1 for x in c))
    S = [x for x in c if x >> best_b & 1]
    assert len(S) == Solution().largestCombination(c) and reduce(operator.and_, S) > 0
print("tập chứng minh (witness) hợp lệ")

tập chứng minh (witness) hợp lệ


### 12.4 Cực trị theo ràng buộc

- `10⁵` số đều bằng `10⁷` ⇒ đáp án `10⁵`.
- `10⁵` số là các lũy thừa của 2 xoay vòng `1, 2, 4, …, 2²³` ⇒ mỗi bit có `⌈10⁵/24⌉` hoặc `⌊10⁵/24⌋` số.

In [24]:
same = [10**7] * 10**5
pw = [1 << (i % 24) for i in range(10**5)]
for name, f in ALL.items():
    if name == "brute": continue
    assert f(same) == 10**5
    assert f(pw) == -(-10**5 // 24)
print("cực trị OK:", -(-10**5 // 24))

cực trị OK: 4167


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ xét bit cao nhất của `max(candidates)`

In [25]:
def wrong_top_bit_only(candidates):
    b = max(candidates).bit_length() - 1
    return sum(x >> b & 1 for x in candidates)                 # SAI: bỏ qua các bit khác

print("bản sai :", wrong_top_bit_only([64, 3, 3, 3]), "| bản đúng:", Solution().largestCombination([64, 3, 3, 3]))

bản sai : 1 | bản đúng: 3


**Sửa:** duyệt **mọi** bit.

### Lỗi 2 — Tham lam thêm dần (phụ thuộc thứ tự)

In [26]:
def wrong_greedy(candidates):
    cur = -1; size = 0
    for x in candidates:
        if cur & x:
            cur &= x; size += 1
    return size

c = [6, 1, 3, 5, 7]
print("bản sai :", wrong_greedy(c), "| bản đúng:", Solution().largestCombination(c))

bản sai : 3 | bản đúng: 4


**Vì sao sai:** lấy `6` trước khóa tập vào các bit {1, 2}, loại mất các số lẻ — trong khi bit 0 có tới 4 số (`1, 3, 5, 7`). Lựa chọn sớm "tham lam" không thể hoàn tác. **Sửa:** đếm theo từng bit.

### Lỗi 3 — Nhầm AND thành OR (đếm số khác 0)

In [27]:
def wrong_or(candidates):
    return sum(1 for x in candidates if x > 0)                 # SAI: đó là điều kiện cho OR > 0

print("bản sai :", wrong_or([1, 2, 4, 8]), "| bản đúng:", Solution().largestCombination([1, 2, 4, 8]))

bản sai : 4 | bản đúng: 1


**Sửa:** AND > 0 cần bit **chung cho tất cả**.

### Lỗi 4 — Độ ưu tiên toán tử khi viết `x & 1 << b`

In [28]:
x, b = 6, 1
print("x & 1 << b    =", x & 1 << b, "(= x & (1 << b), Python: << mạnh hơn &)")
print("(x & 1) << b  =", (x & 1) << b, "(khác nghĩa!)")
def wrong_precedence(candidates):
    return max(sum(1 for x in candidates if (x & 1) << b) for b in range(24))    # SAI: chỉ xét bit 0

print("bản sai :", wrong_precedence([2, 6, 10, 1]), "| bản đúng:", Solution().largestCombination([2, 6, 10, 1]))

x & 1 << b    = 2 (= x & (1 << b), Python: << mạnh hơn &)
(x & 1) << b  = 0 (khác nghĩa!)
bản sai : 1 | bản đúng: 3


**Sửa:** viết rõ `x & (1 << b)` hoặc `(x >> b) & 1`.

### Lỗi 5 — Giới hạn số bit quá nhỏ (ví dụ chỉ 8 hay 16 bit)

In [29]:
def wrong_few_bits(candidates):
    return max(sum(x >> b & 1 for x in candidates) for b in range(16))          # SAI: 10^7 cần 24 bit

c = [2**20] * 4 + [1, 1]
print("bản sai :", wrong_few_bits(c), "| bản đúng:", Solution().largestCombination(c))

bản sai : 2 | bản đúng: 4


**Sửa:** `range(24)` hoặc `range(max(candidates).bit_length())`.

### Lỗi 6 — Chọn các số có **nhiều bit 1 nhất** (popcount lớn)

In [30]:
def wrong_popcount(candidates):
    # SAI: lấy các số có nhiều bit 1 nhất rồi đếm, nghĩ rằng "giàu bit" thì dễ có bit chung
    k = max(bin(x).count("1") for x in candidates)
    S = [x for x in candidates if bin(x).count("1") == k]
    return len(S) if reduce(operator.and_, S) > 0 else 1

c = [7, 7 << 3, 7 << 6, 1, 1, 1, 1]
print("bản sai :", wrong_popcount(c), "| bản đúng:", Solution().largestCombination(c))

bản sai : 1 | bản đúng: 5


**Vì sao sai:** số nhiều bit 1 không đảm bảo có bit **chung** với nhau (`7, 56, 448` rời nhau), trong khi bốn số `1` cùng bit 0. Điều quan trọng là **cột** bit, không phải **hàng**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài về AND/OR/XOR trên tập con ⇒ nghĩ **theo từng bit độc lập** (mỗi cột bit là một bài con).
- "AND > 0" ⇔ "có bit chung"; "OR ≥ …" ⇔ đếm bit hợp; "XOR" ⇔ chẵn lẻ số bit.
- Số ≤ 10⁷ ⇒ 24 bit; ≤ 10⁹ ⇒ 30 bit; ≤ 2⁶³ ⇒ 64 bit.
- Mẹo `x & −x` lấy bit thấp nhất; `x & (x − 1)` tắt bit thấp nhất.
- Khi đề hỏi "tập con lớn nhất/nhỏ nhất thỏa điều kiện bit", hãy thử viết điều kiện dưới dạng "**tồn tại bit**" hoặc "**mọi bit**" — thường sẽ tách được thành các bài con theo cột.
- Luôn có "tập chứng minh" (witness): tìm được bit tốt nhất thì in luôn tập — vừa để kiểm tra, vừa trả lời biến thể "xuất tập".
- Bài cùng họ: 191 Number of 1 Bits, 338 Counting Bits, 477 Total Hamming Distance (đếm theo cột bit), 2411 Smallest Subarrays With Maximum Bitwise OR, 1863 Sum of All Subset XOR Totals.

## 15. Biến thể của bài

### 15.1 Biến thể A — Tổ hợp lớn nhất có AND **≥ k** (k > 0) — chỉ cho n nhỏ, so với đếm theo "mặt nạ bao"

AND ≥ k không còn tách theo bit đơn. Với n nhỏ dùng vét cạn; với n lớn có thể duyệt các "tiền tố bit" của k (giống bài XOR lớn nhất) — ở đây minh họa vét cạn + một cận trên đơn giản `cnt[bit cao nhất của k]`.

In [31]:
def largest_and_at_least_brute(c, k):
    n = len(c); best = 0
    for mask in range(1, 1 << n):
        a = -1; s = 0
        for i in range(n):
            if mask >> i & 1: a &= c[i]; s += 1
        if a >= k: best = max(best, s)
    return best

def largest_and_at_least(c, k):
    # tập có AND >= k: với mỗi "mẫu" m = (tiền tố của k) | bit 1 ở vị trí k có bit 0, hoặc m = k:
    # lấy mọi x chứa m (x & m == m) -> AND chứa m -> AND >= m >= k
    cands = [k]
    for b in range(24):
        if not (k >> b & 1):
            cands.append(((k >> b) | 1) << b)          # giữ bit cao của k, bật bit b, xóa bit thấp
    return max(sum(1 for x in c if x & m == m) for m in cands)

rng = random.Random(151)
for _ in range(300):
    c = [rng.randint(1, 31) for _ in range(rng.randint(1, 10))]; k = rng.randint(1, 31)
    assert largest_and_at_least(c, k) == largest_and_at_least_brute(c, k), (c, k)
print(largest_and_at_least([16, 17, 71, 62, 12, 24, 14], 8), "- khớp brute force")

4 - khớp brute force


Vì sao đúng? AND ≥ k ⇔ AND **bằng** k hoặc tại bit cao nhất mà AND khác k, AND có 1 còn k có 0 (so sánh nhị phân từ bit cao xuống). Mỗi trường hợp ứng với một mẫu `m` mà AND phải chứa — và tập lớn nhất chứa mẫu `m` là mọi `x ⊇ m`.

### 15.2 Biến thể B — Tổ hợp lớn nhất có AND **bằng đúng** một giá trị `t` (t > 0)

Mọi phần tử phải chứa `t` (`x & t == t`); lấy **tất cả** chúng — AND của chúng ⊇ t; nếu AND > t (thừa bit) thì không có tập nào bằng đúng t (bớt phần tử chỉ làm AND tăng/giữ). Kiểm chứng:

In [32]:
def largest_and_equal(c, t):
    S = [x for x in c if x & t == t]
    return len(S) if S and reduce(operator.and_, S) == t else 0

def largest_and_equal_brute(c, t):
    n = len(c); best = 0
    for mask in range(1, 1 << n):
        a = -1; s = 0
        for i in range(n):
            if mask >> i & 1: a &= c[i]; s += 1
        if a == t: best = max(best, s)
    return best

rng = random.Random(152)
for _ in range(300):
    c = [rng.randint(1, 15) for _ in range(rng.randint(1, 9))]; t = rng.randint(1, 15)
    assert largest_and_equal(c, t) == largest_and_equal_brute(c, t)
print(largest_and_equal([16, 17, 71, 62, 12, 24, 14], 16), largest_and_equal([3, 7, 11], 3), "- khớp brute force")

4 3 - khớp brute force


### 15.3 Biến thể C — LC 477: tổng khoảng cách Hamming mọi cặp (cùng kỹ thuật đếm theo cột bit)

Ở mỗi bit `b`, có `cnt` số có bit 1 và `n − cnt` số có bit 0 ⇒ đóng góp `cnt·(n − cnt)` cặp khác nhau.

In [33]:
def total_hamming(nums):
    n = len(nums)
    return sum((c := sum(x >> b & 1 for x in nums)) * (n - c) for b in range(32))

nums = [4, 14, 2]
print(total_hamming(nums), sum(bin(a ^ b).count("1") for i, a in enumerate(nums) for b in nums[i + 1:]))

6 6


### 15.4 Biến thể D — **Mảng con liên tiếp** dài nhất có AND > 0

Bây giờ phải liên tiếp. Với mỗi bit `b`, mảng con liên tiếp dài nhất mà mọi phần tử có bit `b` là "chuỗi 1 dài nhất" trên cột `b` ⇒ đáp án = max qua các bit. O(n·B).

In [34]:
def longest_subarray_and_positive(nums):
    best = 0
    for b in range(24):
        run = 0
        for x in nums:
            run = run + 1 if x >> b & 1 else 0
            best = max(best, run)
    return best

def longest_subarray_brute(nums):
    n = len(nums); best = 0
    for i in range(n):
        a = -1
        for j in range(i, n):
            a &= nums[j]
            if a > 0: best = max(best, j - i + 1)
            else: break
    return best

rng = random.Random(154)
for _ in range(500):
    a = [rng.randint(1, 63) for _ in range(rng.randint(1, 20))]
    assert longest_subarray_and_positive(a) == longest_subarray_brute(a)
print(longest_subarray_and_positive([16, 17, 71, 62, 12, 24, 14]), "(tổ hợp tùy ý cho 4) - khớp brute force")

4 (tổ hợp tùy ý cho 4) - khớp brute force


Chú ý sự khác biệt: tổ hợp tùy ý chỉ cần **đếm** số phần tử có bit `b`; mảng con liên tiếp cần **chuỗi liên tiếp dài nhất** có bit `b`.

### 15.5 Biến thể E — Số phần tử **ít nhất phải bỏ** để AND của phần còn lại > 0

Bằng `n − largestCombination(candidates)` — vì giữ lại nhiều nhất ⇔ bỏ ít nhất.

In [35]:
def min_removals(candidates):
    return len(candidates) - Solution().largestCombination(candidates)
print(min_removals([16, 17, 71, 62, 12, 24, 14]), min_removals([8, 8]), min_removals([1, 2, 4, 8]))

3 0 3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi tập con | đúng định nghĩa | O(2ⁿ·n) | O(1) | kiểm chứng |
| 2. Theo bit | 24 lượt đếm | O(n·B) | O(1) | **mặc định** |
| 3. Theo số, bit bật | `x & −x` | O(số bit 1) | O(B) | số thưa bit |
| 4. Chuỗi nhị phân | `bin(x)` | O(n·B) | O(B) | dễ đọc |
| 5. Mặt nạ + generator | `x & (1 << b)` | O(n·B) | O(1) | hằng số tốt |
| 6. Counter trước | giá trị lặp | O(n + D·B) | O(D) | nhiều trùng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `largestCombination([33, 93, 31, 99, 74, 37, 3, 4, 2, 94, 77, 10, 75, 54, 24, 95, 65, 100, 41, 82, 35, 65, 38, 49, 85, 72, 67, 21, 20, 31])` — chỉ cần chạy; và `largestCombination([7, 7, 7, 1])`?

**Bài 2 (tính tay):** Với `[5, 9, 12, 13]`, lập bảng `cnt` cho bit 0..3.

**Bài 3 (điền TODO):** Viết `bt3_todo(candidates)` trả về **một tổ hợp** (danh sách giá trị) kích thước lớn nhất có AND > 0.

**Bài 4 (tự viết lại):** Viết `bt4_todo(candidates)` cài cách 3 (theo số, chỉ duyệt bit bật) không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(candidates)` trả về **số lượng** tổ hợp (tập con khác rỗng, theo chỉ số) có AND > 0, cho `n ≤ 20`, dùng **bao hàm–loại trừ** trên tập bit: số tập có AND chứa ít nhất một bit = Σ_{T ≠ ∅} (−1)^{|T|+1} (2^{cnt(T)} − 1), với `cnt(T)` = số phần tử chứa mọi bit trong `T` (giới hạn giá trị ≤ 255 để số bit nhỏ).

In [36]:
def bt3_todo(candidates):
    # TODO: một tổ hợp lớn nhất có AND > 0
    pass

def bt4_todo(candidates):
    # TODO: theo số, chỉ duyệt bit bật
    pass

def bt5_todo(candidates):
    # TODO: đếm số tập con có AND > 0 bằng bao hàm-loại trừ
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: chạy `Solution`. Mức 2: `[7,7,7,1]`: bit 0 có cả 4 số. Mức 3: `4`.

**Bài 2.** Mức 1: `5 = 0101, 9 = 1001, 12 = 1100, 13 = 1101`. Mức 2: bit 0: 5, 9, 13 ⇒ 3; bit 2: 5, 12, 13 ⇒ 3. Mức 3: `cnt = [3, 0, 3, 3]` ⇒ đáp án 3.

**Bài 3.** Mức 1: tìm bit `b` có `cnt` lớn nhất. Mức 2: lọc `x >> b & 1`. Mức 3: trả list đó.

**Bài 4.** Mức 1: `low = x & −x`. Mức 2: chỉ số bit `low.bit_length() − 1`. Mức 3: `x ^= low` tới khi `x = 0`.

**Bài 5.** Mức 1: "AND có bit b" là sự kiện `A_b`; cần `|∪ A_b|`. Mức 2: `|∩_{b∈T} A_b|` = số tập con khác rỗng của các phần tử chứa mọi bit trong `T` = `2^{cnt(T)} − 1`. Mức 3: duyệt mọi `T ⊆ {bit 0..7}` khác rỗng, cộng/trừ theo `|T|`.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** chạy ở cell dưới; `[7, 7, 7, 1]` → `4`. **Bài 2:** `cnt = [3, 0, 3, 3]` ⇒ `3`.

In [37]:
big1 = [33, 93, 31, 99, 74, 37, 3, 4, 2, 94, 77, 10, 75, 54, 24, 95, 65, 100, 41, 82, 35, 65, 38, 49, 85, 72, 67, 21, 20, 31]
print("Bài 1:", Solution().largestCombination(big1))
assert Solution().largestCombination([7, 7, 7, 1]) == 4
assert [sum(x >> b & 1 for x in [5, 9, 12, 13]) for b in range(4)] == [3, 0, 3, 3]

def bt3(candidates):
    b = max(range(24), key=lambda b: sum(x >> b & 1 for x in candidates))
    return [x for x in candidates if x >> b & 1]

def bt4(candidates):
    return per_number(candidates)

def bt5(candidates):
    total = 0
    for T in range(1, 256):
        cnt = sum(1 for x in candidates if x & T == T)
        sign = 1 if bin(T).count("1") % 2 else -1
        total += sign * ((1 << cnt) - 1)
    return total

def bt5_brute(candidates):
    n = len(candidates); c = 0
    for mask in range(1, 1 << n):
        a = -1
        for i in range(n):
            if mask >> i & 1: a &= candidates[i]
        c += a > 0
    return c

rng = random.Random(1975)
for _ in range(300):
    c = [rng.randint(1, 255) for _ in range(rng.randint(1, 10))]
    S = bt3(c)
    assert len(S) == brute(c) and reduce(operator.and_, S) > 0
    assert bt4(c) == brute(c)
    assert bt5(c) == bt5_brute(c)
print("Đáp án bài tập: tất cả assert OK")

Bài 1: 18


Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
ans = 0
for b in range(24):                         # 10^7 < 2^24
    ans = max(ans, sum(1 for x in candidates if x >> b & 1))
return ans
```

**Checklist:**
- [ ] AND > 0 ⇔ có **một bit chung** cho mọi phần tử.
- [ ] Xét **mọi** bit (24 với ràng buộc này).
- [ ] Không tham lam thêm dần.
- [ ] Ngoặc rõ ràng khi viết biểu thức bit (`x & (1 << b)`).
- [ ] Đáp án luôn ≥ 1 (mọi số dương có ít nhất một bit).
- [ ] Đã thử: mọi số là lũy thừa 2 khác nhau (đáp án 1), số trùng nhau, ví dụ `[6, 5, 3]`.